In [255]:
def build_propositions(stages=3):
    terms = ["E"]

    for _ in range(stages):
        new_terms = []

        for term in terms:
            for i, y in enumerate(term):
                if y == "E":
                    new_term = term[:i] + "JEE" + term[i+1:]

                    if new_term not in new_terms and new_term not in terms:
                        new_terms.append(new_term)

        terms.extend(new_terms)

    elementof_op = ["∈"]
    eq_op = ["="]

    WFF = []
    for i in terms:
        for y in terms:
            if y == i:
                continue
            AFO = [i] + elementof_op + [y]
            AFE = [i] + eq_op + [y]

            WFF.append(AFE)
            WFF.append(AFO)

    Propositions = []

    for i in WFF:
        for y in WFF:
            if y == i:
                continue

            Proposition_or = ["or"] + [i] + [y]
            Proposition_and = ["and"] + [i] + [y]
            Proposition_imply = ["imp"] + [i] + [y]

            Propositions.append(Proposition_or)
            Propositions.append(Proposition_and)
            Propositions.append(Proposition_imply)

        Proposition_not = ["not"] + [i]
        Propositions.append(Proposition_not)

    return Propositions, WFF

def all_citations(i):
    cites = [[]]
    for prev in range(i):
        new = []
        for c in cites:
            new.append(c + [prev])
        cites = cites + new
    return cites


def all_lines(i, Propositions, WFF):
    rules = ["axiom", "taut"]
    lines = []

    for f in WFF:
        for c in all_citations(i):
            lines.append(["atom", f, c])


    for f in Propositions:
        for r in rules:
            for c in all_citations(i):
                if r == "taut" and c == []:
                    continue
                lines.append([f, r, c])


    return lines


def all_proofs(i, Propositions, WFF, limit=1000):
    if i == 0:
        return [[]]

    smaller_proofs = all_proofs(i - 1, Propositions, WFF, limit)
    proofs = []

    for p in smaller_proofs:
        possible_lines = all_lines(len(p), Propositions, WFF)

        for line in possible_lines:
            new_proof = p + [line]
            proofs.append(new_proof)

            if len(proofs) >= limit:
                return proofs

    return proofs

def collect_E_atoms(x):
    atoms = []

    def helper(node):
        if isinstance(node, list):
            if len(node) == 3 and not isinstance(node[0], list) and not isinstance(node[2], list):
                atoms.append([node])
                return
            if isinstance(node[0], list):
                helper(node[0])
            if isinstance(node[2], list):
                helper(node[2])

    if isinstance(x, list) and len(x) > 0 and isinstance(x[0], list):
        for item in x:
            helper(item)
    else:
        helper(x)

    flat = []
    for a in atoms:
        flat.extend(a)

    result = []
    for v in flat:
        if v not in result:
            result.append(v)

    return result


def proof_checker(line, proof):
    y = line
    citations = y[2]
    z = collect_E_atoms([y[0]])

    cited_lines = []
    if citations:
        cited_lines = [proof[cite][0] for cite in citations]

    if y[1] == "taut":
        rows = []
        atoms = z + cited_lines
        N = len(atoms)

        for i in range(2**N):
            row = []

            for j in range(N):
                bit = (i >> (N - j - 1)) & 1

                if citations and atoms[j] in cited_lines:
                    bit = 1

                row.append([atoms[j], bit])

            rows.append(row)

        final_bits = []

        left_atom = y[0][1]
        right_atom = y[0][2]
        op = y[0][0]

        for row in rows:
            firstboolean = None
            secondboolean = None

            for atom, bit in row:
                if atom == left_atom:
                    firstboolean = bit
                if atom == right_atom:
                    secondboolean = bit

            if op == "or":
                result = 1 if (firstboolean or secondboolean) else 0
            elif op == "and":
                result = 1 if (firstboolean and secondboolean) else 0
            elif op == "imp":
                result = 1 if ((not firstboolean) or secondboolean) else 0
            elif op == "not":
                result = 1 if (not firstboolean) else 0

            final_bits.append(result)

            if result == 0:
                return False

In [253]:
Propositions, WFF = build_propositions()

x = all_proofs(3,Propositions,WFF, 100000)

In [257]:
x[0]

[['atom', ['E', '=', 'JEE'], []],
 ['atom', ['E', '=', 'JEE'], []],
 ['atom', ['E', '=', 'JEE'], []]]